# Simple PDF RAG with Groq

This notebook is a simplified teaching version of `rag1_simple_pdf.py`.

Assumptions:
- you already have a searchable PDF such as `sample_ocr.pdf`
- you want to use Groq only
- you want a clean notebook for beginners

In [ ]:
!pip -q install openai python-dotenv numpy pypdf tiktoken fastembed

In [ ]:
import os
from getpass import getpass

import numpy as np
import tiktoken
from fastembed import TextEmbedding
from openai import OpenAI
from pypdf import PdfReader

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    pdf_file_path = next(iter(uploaded.keys())) if uploaded else "sample_ocr.pdf"
except ImportError:
    pdf_file_path = input("PDF path (default sample_ocr.pdf): ").strip() or "sample_ocr.pdf"

groq_api_key = os.getenv("GROQ_API_KEY") or getpass("Enter GROQ_API_KEY: ").strip()
chat_model = os.getenv("GROQ_CHAT_MODEL") or "qwen/qwen3.8-27b"
user_question = input("Question (default: What are the main findings in this document?): ").strip() or "What are the main findings in this document?"

client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("PDF:", pdf_file_path)
print("Model:", chat_model)
print("Question:", user_question)

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    text_parts = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_parts.append(page_text.strip())
    return "\n".join(text_parts)

def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> list[str]:
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    step = chunk_size - chunk_overlap
    chunks = []
    for start in range(0, len(tokens), step):
        chunk_tokens = tokens[start:start + chunk_size]
        chunks.append(tokenizer.decode(chunk_tokens))
    return chunks

def get_embedding(text: str) -> list[float]:
    return list(next(embedder.embed([text])))

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def retrieve_best_chunk(chunks: list[str], question: str) -> str:
    chunk_embeddings = [get_embedding(chunk) for chunk in chunks]
    question_embedding = get_embedding(question)
    similarities = [cosine_similarity(question_embedding, chunk_embedding) for chunk_embedding in chunk_embeddings]
    best_chunk_index = int(np.argmax(similarities))
    return chunks[best_chunk_index]

def ask_groq(retrieved_context: str, question: str) -> str:
    completion = client.chat.completions.create(
        model=chat_model,
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant. Answer the user's question using only the context below.\n\n" + retrieved_context,
            },
            {
                "role": "user",
                "content": question,
            },
        ],
    )
    return completion.choices[0].message.content or ""

In [ ]:
print(f"Reading {pdf_file_path}...")
raw_pdf_text = extract_text_from_pdf(pdf_file_path)
if not raw_pdf_text.strip():
    raise ValueError("This notebook expects a searchable PDF like sample_ocr.pdf.")

print("Chunking PDF text...")
chunks = chunk_text(raw_pdf_text)
print(f"Created {len(chunks)} chunks")

print("Retrieving the most relevant chunk...")
retrieved_context = retrieve_best_chunk(chunks, user_question)

print(f"Sending the final prompt to Groq via {chat_model}...")
answer = ask_groq(retrieved_context, user_question)

print("\n--- Answer ---")
print(answer)